# state-dict-load — worked example 1: load_state_dict(strict=False) returns missing/unexpected

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `state-dict-load`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`load_state_dict(ckpt, strict=False)` does not raise on key mismatches; it returns an `_IncompatibleKeys(missing_keys, unexpected_keys)` named tuple. `missing_keys` are keys the model has but the checkpoint lacked; `unexpected_keys` are checkpoint keys the model doesn't want. This is the escape hatch for loading a partial or head-swapped checkpoint.

## Worked solution

We define a `ToyNet` with a `backbone` linear and an `fc` head. We build a checkpoint that contains the backbone weights (matching) plus a stray `extra` key, but OMITS the `fc` keys. Calling `load_state_dict(ckpt, strict=False)` loads what it can and reports the rest: `fc.weight`/`fc.bias` show up as missing (model has them, ckpt didn't), and `extra` shows up as unexpected (ckpt has it, model doesn't). We print both lists to confirm the classification.

In [ ]:
import torch.nn as nn


class ToyNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = nn.Linear(10, 16)
        self.fc = nn.Linear(16, n_classes)
    def forward(self, x):
        return self.fc(self.backbone(x))


t.manual_seed(0)
model = ToyNet(7)
ckpt = {
    'backbone.weight': t.randn(16, 10),
    'backbone.bias': t.randn(16),
    'extra': t.randn(3),
}
result = model.load_state_dict(ckpt, strict=False)
print('missing:', sorted(result.missing_keys))
print('unexpected:', sorted(result.unexpected_keys))